# Near-field ptychography with an X-ray waveguide

This example reconstructs a near-field ptychography scan of a test object (the IRP logo, Eulitha test pattern) recorded at the GINIX endstation of beamline P10 (PETRA III, DESY).
The sample is placed a few mm behind an X-ray waveguide, which acts as a quasi point source. The detector is 5 m downstream, so the scan is recorded in the holographic (near-field) regime at high geometric magnification.

**Data**

1. Download the raw detector file `Eulitha_v3_R3_objects_01_00000.h5` from the Göttingen Research Online repository: https://data.goettingen-research-online.de/dataset.xhtml?persistentId=doi:10.25625/IM01EA
2. The file `p10_waveguide_metadata.h5` next to this notebook contains everything the raw file does not: scan positions, bad-pixel mask and geometry.

**What the notebook does**

1. Extracts the 121 ptychography frames from the raw Eiger file, fills the defective detector pixels by inpainting and writes a small ptypy-ready HDF5 file.
2. Reconstructs the scan in near-field geometry on the GPU with three algorithms: difference map (DM), ePIE and maximum likelihood (ML) with the wavefield preconditioner.
3. Plots the object phase and the complex probe for each algorithm and reports performance metrics.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # select the GPU

import time
import h5py
import hdf5plugin  # registers the LZ4 filter needed to read the raw Eiger frames
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import ptypy
import ptypy.utils as u

In [ ]:
# ---- paths: adjust to your system ----
raw_file = "..path/to//Eulitha_v3_R3_objects_01_00000.h5"      # downloaded from doi:10.25625/IM01EA
metadata_file = "..path/to/p10_waveguide_metadata.h5"          # shipped with this notebook
prepared_file = "..path/to//p10_waveguide_ptycho.h5"            # written below, input for ptypy
recon_file = "..path/to/p10_waveguide_recon.ptyr"

## 1. Extract the ptychography frames

The raw file holds a single frame series `entry/data/data_000000` with 557 frames, in the order they were recorded during the beamtime. The ptychographic scan (an 11 × 11 mesh with 1 µm steps) is **frames 20 to 140**; the frame indices are stored in `metadata['frames']`.

Two kinds of invalid pixels are treated differently:

- **Module gaps** (the 37-pixel rows and 10-pixel column between the Eiger modules) carry the value `2**32 - 1`. They hold no signal, so they are set to 0 and excluded from the reconstruction with a mask.
- **Defective pixels** (flagged in the mask from the metadata file, `True` = valid pixel) are isolated, so they are filled by inpainting from their neighbours (`cv2.inpaint`, Navier-Stokes method). They then take part in the reconstruction like any other pixel.

To keep the input file small, the frames are cropped to a 779 × 779 pixel region around the beam.

In [ ]:
data_center = (1385, 678)  # (row, column) of the beam on the detector
data_shape = (779, 779)
crop = np.s_[data_center[0] - data_shape[0] // 2:data_center[0] - data_shape[0] // 2 + data_shape[0],
             data_center[1] - data_shape[1] // 2:data_center[1] - data_shape[1] // 2 + data_shape[1]]

def get_eiger_gaps():
    """Eiger 4M module gaps on the full detector (2167, 2070), True = valid pixel."""
    gaps = np.zeros((2167, 2070), dtype=bool)
    module_size = (514, 1030)
    gap_size = (37, 10)
    for i_gap in range(3):
        start = module_size[0] + i_gap * (module_size[0] + gap_size[0])
        gaps[start:start + gap_size[0]] = True
    gaps[:, module_size[1]:module_size[1] + gap_size[1]] = True
    return ~gaps

use_refined_positions = True  # False: motor positions as recorded at the beamline. True: positions already refined

with h5py.File(metadata_file, "r") as meta:
    frames = meta["frames"][:]
    valid_pixels = meta["mask"][crop]  # False at module gaps and defective pixels
    suffix = "" if use_refined_positions else "_original"
    positions_z = meta["positions_z" + suffix][:]  # mm
    positions_y = meta["positions_y" + suffix][:]  # mm
    energy = meta["pars/energy"][()]  # keV
    detector_pixel = meta["pars/px"][()]  # m
    z01 = meta["pars/z01"][()]  # m, waveguide exit -> sample
    z02 = meta["pars/z02"][()]  # m, waveguide exit -> detector

with h5py.File(raw_file, "r") as raw:
    dataset = raw["entry/data/data_000000"]
    images = np.empty((len(frames), *data_shape), dtype=np.float32)
    for i, frame in enumerate(frames):
        images[i] = dataset[frame][crop]

mask = get_eiger_gaps()[crop]      # True = valid; only the module gaps are masked in ptypy
defects = mask & ~valid_pixels     # defective pixels outside the gaps -> inpainted
images[:, ~valid_pixels] = 0       # module gaps (2**32 - 1) and defective pixels
for i in range(len(images)):
    images[i] = cv2.inpaint(images[i], defects.astype(np.uint8), 2, cv2.INPAINT_NS)
print(f"inpainted {defects.sum()} defective pixels per frame, {(~mask).sum()} gap pixels masked")

with h5py.File(prepared_file, "w") as f:
    f["images"] = images
    f["mask"] = mask.astype(np.uint8)
    f["positions_z"] = positions_z
    f["positions_y"] = positions_y

print(f"{images.shape[0]} frames of shape {images.shape[1:]} written to {prepared_file}")

## 2. Geometry


In [ ]:
wavelength = 12.39842 / energy * 1e-10  # m
z12 = z02 - z01
magnification = z02 / z01
effective_pixel = detector_pixel / magnification
effective_distance = z12 / magnification
fresnel_number = effective_pixel**2 / (wavelength * effective_distance)

print(f"Energy:                  {energy:.2f} keV (wavelength {wavelength:.3e} m)")
print(f"Magnification:           {magnification:.0f}")
print(f"Effective pixel size:    {effective_pixel * 1e9:.2f} nm")
print(f"Effective distance:      {effective_distance * 1e3:.3f} mm")
print(f"Fresnel number (pixel):  {fresnel_number:.2e}")

Quick look at the data: a single hologram, the mean over all frames (used to initialise the probe) and the scan positions.

In [ ]:
flat = images.mean(axis=0)
flat[~mask] = np.median(flat[mask])  # no holes in the initial probe

fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
im = ax[0].imshow(np.ma.masked_array(images[60], ~mask), cmap="gray", norm=LogNorm())
ax[0].set_title("Hologram, frame 60")
fig.colorbar(im, ax=ax[0])
im = ax[1].imshow(flat, cmap="gray", norm=LogNorm())
ax[1].set_title("Mean of all frames")
fig.colorbar(im, ax=ax[1])
ax[2].plot(positions_y * 1e3, positions_z * 1e3, "-o", markersize=3)
ax[2].plot(positions_y[0] * 1e3, positions_z[0] * 1e3, "ro", label="start")
ax[2].set_aspect("equal")
ax[2].set_xlabel("y [µm]")
ax[2].set_ylabel("z [µm]")
ax[2].set_title("Scan positions")
ax[2].legend()
plt.tight_layout()
plt.show()




## 3. ptypy reconstruction


In [ ]:
ptypy.load_ptyscan_module("hdf5_loader")
ptypy.load_gpu_engines("cupy")

engines = ["dm"]  # reconstruction algorithms to run and compare

def make_params(engine):
    """Parameter tree for one reconstruction with the given engine ('dm', 'epie' or 'ml')."""
    p = u.Param()
    p.verbose_level = "interactive"
    p.run = f"p10_waveguide_{engine}"
    p.frames_per_block = 200  # all 121 frames in one block

    p.io = u.Param()
    p.io.rfile = os.path.splitext(recon_file)[0] + f"_{engine}.ptyr"
    p.io.home = os.path.dirname(os.path.abspath(recon_file))
    p.io.benchmark = "all"  # time data loading and engine steps, see Performance metrics below
    p.io.interaction = u.Param(active=False)
    p.io.autosave = u.Param(active=False)
    p.io.autoplot = u.Param(active=False)

    p.scans = u.Param()
    p.scans.scan_00 = u.Param()
    # projection engines (DM) use BlockFull, gradient-based and stochastic engines use BlockGradFull
    p.scans.scan_00.name = "BlockFull" if engine == "dm" else "BlockGradFull"
    p.scans.scan_00.propagation = "nearfield"

    # data
    p.scans.scan_00.data = u.Param()
    p.scans.scan_00.data.name = "Hdf5Loader"
    p.scans.scan_00.data.intensities = u.Param(file=prepared_file, key="images")
    p.scans.scan_00.data.mask = u.Param(file=prepared_file, key="mask")  # 1 = valid pixel
    p.scans.scan_00.data.positions = u.Param()
    p.scans.scan_00.data.positions.file = prepared_file
    p.scans.scan_00.data.positions.slow_key = "positions_z"
    p.scans.scan_00.data.positions.slow_multiplier = 1e-3  # mm -> m
    p.scans.scan_00.data.positions.fast_key = "positions_y"
    p.scans.scan_00.data.positions.fast_multiplier = 1e-3  # mm -> m
    p.scans.scan_00.data.shape = data_shape
    p.scans.scan_00.data.center = (data_shape[0] // 2, data_shape[1] // 2)  # frames are already centred
    p.scans.scan_00.data.auto_center = False
    p.scans.scan_00.data.orientation = 0
    p.scans.scan_00.data.energy = energy
    p.scans.scan_00.data.psize = effective_pixel
    p.scans.scan_00.data.distance = effective_distance

    # initial probe: amplitude of the mean frame, flat phase
    p.scans.scan_00.illumination = u.Param()
    p.scans.scan_00.illumination.model = np.sqrt(flat).astype(np.complex64)
    p.scans.scan_00.illumination.aperture = None

    # initial object: empty (transmission 1)
    p.scans.scan_00.sample = u.Param()
    p.scans.scan_00.sample.fill = 1
    p.scans.scan_00.sample.process = None
    p.scans.scan_00.sample.diversity = None

    p.scans.scan_00.coherence = u.Param(num_probe_modes=1)

    # reconstruction engine
    p.engines = u.Param()
    e = p.engines.engine1 = u.Param()
    e.numiter = 300
    e.numiter_contiguous = 10
    e.probe_support = 0.5
    e.probe_update_start = 0
    if engine == "dm":
        # difference map with a pure phase object
        e.name = "DM_cupy"
        e.alpha = 0.75  # 1 = DM, 0 = alternating projections
        e.clip_object = (1 - 1e-6, 1 + 1e-6)  # pure phase object
    elif engine == "epie":
        # extended ptychographic iterative engine (stochastic, one view at a time)
        e.name = "EPIE_cupy"
        e.alpha = 1.0  # object update step size
        e.beta = 1.0   # probe update step size
        e.object_norm_is_global = True
        e.compute_log_likelihood = True
    elif engine == "ml":
        # maximum likelihood (Gaussian noise model) with the wavefield preconditioner
        e.name = "ML_cupy"
        e.ML_type = "Gaussian"
        e.wavefield_precond = True
        e.wavefield_delta_object = 0.1
        e.wavefield_delta_probe = 0.1
        e.reg_del2 = False
        e.floating_intensities = False

    # optional: refine the positions
    # e.position_refinement = u.Param()
    # e.position_refinement.method = "Annealing"
    # e.position_refinement.start = 20
    # e.position_refinement.stop = 250
    # e.position_refinement.interval = 10
    # e.position_refinement.nshifts = 8
    return p

In [ ]:
import gc, json, socket
import cupy as cp

results = {}
for engine in engines:
    p = make_params(engine)
    t0 = time.time()
    P = ptypy.core.Ptycho(p, level=5)
    print(f"{engine}: reconstruction took {(time.time() - t0) / 60:.1f} min")

    results[engine] = dict(
        obj=next(iter(P.obj.S.values())).data[0].copy(),
        probe=next(iter(P.probe.S.values())).data[0].copy(),
        numiter=p.engines.engine1.numiter,
    )

    # save the benchmark timings with the host name, as in the other ptypy application examples
    summary_file = f"summary_p10_waveguide_{engine}.json"
    with open(summary_file, "w") as f:
        json.dump({"host": socket.gethostname(), "nprocs": ptypy.utils.parallel.size, "nruns": 1,
                   "benchmark": dict(P.benchmark)}, f)
    print(f"Benchmark written to {summary_file}")

    # free GPU memory before the next engine
    del P
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()

## 4. Results

In [ ]:
from matplotlib.colors import hsv_to_rgb

titles = {"dm": "DM"}

fig, ax = plt.subplots(len(engines), 3, figsize=(16, 5 * len(engines)), squeeze=False)
for row, engine in enumerate(engines):
    obj, probe = results[engine]["obj"], results[engine]["probe"]

    obj_phase = np.angle(obj)
    # colour range from the illuminated centre: without a pure-phase constraint (ePIE, ML)
    # the unconstrained object edges are noisy and would dominate mean +- 3 std
    ny, nx = obj_phase.shape
    centre = obj_phase[ny // 4:3 * ny // 4, nx // 4:3 * nx // 4]
    m, s = centre.mean(), centre.std()

    # zoom on the centre of the object, where the logo is
    half = int(0.11 * min(ny, nx))
    zoom = np.s_[ny // 2 - half:ny // 2 + half, nx // 2 - half:nx // 2 + half]

    im = ax[row, 0].imshow(obj_phase, cmap="gray", vmin=m - 3 * s, vmax=m + 3 * s)
    ax[row, 0].set_title(f"{titles[engine]}: object phase")
    fig.colorbar(im, ax=ax[row, 0], label="rad")
    im = ax[row, 1].imshow(obj_phase[zoom], cmap="gray", vmin=m - 3 * s, vmax=m + 3 * s)
    ax[row, 1].set_title(f"{titles[engine]}: inset")
    fig.colorbar(im, ax=ax[row, 1], label="rad")

    # complex probe: hue = phase, brightness = amplitude
    probe_amp = np.abs(probe)
    amp_norm = np.clip(probe_amp / np.percentile(probe_amp, 99), 0, 1)
    hue = (np.angle(probe) + np.pi) / (2 * np.pi)
    im = ax[row, 2].imshow(hsv_to_rgb(np.stack([hue, amp_norm, amp_norm], axis=-1)))
    ax[row, 2].set_title(f"{titles[engine]}: probe")
    # phase/amplitude colour legend, drawn in the colorbar slot so all panels have the same size
    legend = fig.colorbar(im, ax=ax[row, 2]).ax
    legend.clear()
    phase_grid, amplitude_grid = np.meshgrid(np.linspace(0, 1, 256), np.linspace(0, 1, 256))
    legend.imshow(hsv_to_rgb(np.dstack((phase_grid, amplitude_grid, amplitude_grid))), aspect="auto", origin="lower")
    legend.set_xticks([0, 255]); legend.set_xticklabels(["-π", "π"])
    legend.set_yticks([0, 255]); legend.set_yticklabels(["0", "max"])
    legend.yaxis.tick_right()
    legend.set_xlabel("phase")

for a in ax[:, :3].flat:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
plt.show()

## Performance metrics

In [ ]:
import json
nframes = 121
npixel2 = 779 * 779

for engine in engines:
    niterations = 300
    with open(f"./summary_p10_waveguide_{engine}.json", "r") as f:
        metrics = json.load(f)

    host = metrics["host"]
    total = sum([v for v in metrics["benchmark"].values()])
    time_prep_per_pixel = metrics["benchmark"]["data_load"] / nframes / npixel2 * 1e9
    time_iterate_per_pixel = metrics["benchmark"]["engine_iterate"] / nframes / npixel2 / niterations * 1e9
    print(f"{engine.upper():5s} Total: {total:.0f} s | Preparation: {time_prep_per_pixel:.2f} ns/px | Iterate: {time_iterate_per_pixel:.2f} ns/px")
print(f"Host: {host}")